# GIN — Fixed Architecture, Batch of Feature Configurations (HI-Small)

**Question:** what do manually engineered edge features add when the GNN architecture is held
**exactly fixed**? One session runs a list of configurations through literally the same code:
the `CONFIGS` list in Section 1 is the only thing you edit.

| Knob (per config) | Values | Effect |
|---|---|---|
| `mp` | `none` / `base` / `full` | edge features in message passing; `none` → `GINConv`, else `GINEConv(edge_dim)` |
| `readout` | `base` / `full` / `gfp` | edge features concatenated at the classifier `[h_src, h_dst, e_seed]` |

Fixed for every run (Altman et al. 2023): 2 GIN layers, hidden 64, dropout 0.3, neighbour
sampling `[100, 100]`, `BCEWithLogits(pos_weight=8)`, Adam 1e-3 + cosine, 20 epochs, threshold
chosen on validation, best-val-F1 checkpoint, test scored once. Every run prints and stores the
**architecture-invariant parameter count (17,411)** — it must match across all feature configs.

Per-epoch overfitting diagnostics: train & validation loss (same weighted BCE), F1 (train reuses
the validation threshold), PR-AUC. Each run saves `results.json`, `history.csv`, `best.pt`,
`curves.png` under `outputs/<run_name>/`; a summary table compares all runs at the end.

## 0. Environment (Kaggle)

`LinkNeighborLoader` needs `pyg_lib` (or `torch_sparse`). Adjust the wheel URL to the torch /
CUDA version printed in the first line, then **restart the session** after installing.

In [ ]:
# import torch
# print('torch', torch.__version__, '| cuda', torch.version.cuda)

# # wheel index must match the torch + CUDA version printed above
# !pip install -q torch_geometric
# !pip install -q pyg_lib torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-2.8.0+cu126.html

In [ ]:
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch_geometric

## 1. Imports & configuration

In [ ]:
import json
import os
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.metrics import (average_precision_score, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import GINConv, GINEConv
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch {torch.__version__} | device {device}')

In [ ]:
# ---------------- experiment configurations to run in THIS session ----------------
# Each entry is one experiment. Everything else below is FIXED and shared by all runs.
#   mp      : edge features in message passing   'none' | 'base' | 'full'
#   readout : edge features at the classifier    'base' | 'full' | 'gfp'
CONFIGS = [
    dict(mp='none', readout='base'),   # Run 1 — GIN,  baseline features
    dict(mp='base', readout='base'),   # Run 2 — GINE, baseline features
    dict(mp='none', readout='full'),   # Run 7 — GIN,  +GFP at the readout
    dict(mp='full', readout='full'),   # Run 8 — GINE, +GFP everywhere
]

MP_DIRECTION      = 'in'     # 'in' | 'bidirectional' — held fixed for the whole batch
TEMPORAL_SAMPLING = False    # True samples only edges earlier than the seed (needs recent pyg-lib)

# ---------------- fixed recipe (identical for every run) ----------------
HIDDEN        = 64
NUM_LAYERS    = 2
DROPOUT       = 0.3
NUM_NEIGHBORS = [100, 100]     # neighbours sampled per hop (one entry per GNN layer)
BATCH_SIZE    = 8192           # seed edges per mini-batch
EPOCHS        = 20
LR            = 1e-3
WEIGHT_DECAY  = 1e-5
POS_WEIGHT    = 8.0            # weight of the laundering class in the loss
SEED          = 42

# ---------------- paths ----------------
DATA_DIR = '/kaggle/input/datasets/sandrokhizanishvili/hi-small-gnn'
OUT_ROOT = '/kaggle/working/outputs'

## 2. Load graphs and feature column groups

`feature_meta.json` says which `edge_attr` columns are baseline (first 20) and GFP (next 61).
Each configuration picks its column lists from `COL_SETS`; node features `x` are always used.

In [ ]:
train_graph = torch.load(f'{DATA_DIR}/train_graph.pt', weights_only=False)
val_graph   = torch.load(f'{DATA_DIR}/val_graph.pt',   weights_only=False)
test_graph  = torch.load(f'{DATA_DIR}/test_graph.pt',  weights_only=False)
meta = json.load(open(f'{DATA_DIR}/feature_meta.json'))

all_cols = meta['EDGE_FEAT_COLS']
base_idx = [all_cols.index(c) for c in meta['BASE_EDGE_COLS']]   # 0..19
gfp_idx  = [all_cols.index(c) for c in meta['GFP_FEAT_COLS']]    # 20..80
COL_SETS = {'none': [], 'base': base_idx, 'gfp': gfp_idx, 'full': base_idx + gfp_idx}
NODE_DIM = train_graph.x.shape[1]

for name, g in [('train', train_graph), ('val', val_graph), ('test', test_graph)]:
    n_eval = int(g.eval_mask.sum())
    n_pos  = int((g.y[g.eval_mask] == 1).sum())
    print(f'{name:<5} nodes={g.num_nodes:,} edges={g.edge_index.shape[1]:,} '
          f'evaluated={n_eval:,} laundering={n_pos:,} ({n_pos / n_eval:.4%})')

## 3. Model — one class for every configuration

- `node_proj`: `Linear(node_dim → 64)`
- `NUM_LAYERS` × GIN layer. Each conv uses the **same** node MLP
  `Linear(64,64) → BatchNorm → ReLU → Linear(64,64)` with learnable ε. `mp_edge_dim = 0` → `GINConv`,
  otherwise `GINEConv(edge_dim)` (adds only a `Linear(edge_dim → 64)` on the edge term).
  `bidirectional` adds a second conv over reversed edges, merged with `Linear(128 → 64)`.
  Then ReLU, dropout, and the residual connection `h = h + layer(h)`.
- `classifier`: MLP on `[h_src, h_dst, edge feats]` — hidden 64, one logit.

```text
             node features x  [N x 6]
                      |
             Linear 6->64 . ReLU
                      |
        +-------------v-------------+
        | GIN layer 1               |<-- MP edge features e_uv {0|20|81}
        |  m_u = h_u (+ W_e.e_uv)   |    (absent when MP = 'none' -> GINConv;
        |  (1+eps).h_v + sum m_u    |     bidirectional adds a 2nd conv over
        |  MLP: 64->64.BN.ReLU.     |     reversed edges, concat -> 128->64)
        |       64->64              |
        |  ReLU . Dropout . +resid  |
        +-------------+-------------+
        +-------------v-------------+
        | GIN layer 2 (same form)   |<-- e_uv
        +-------------+-------------+
                      |
             embeddings h  [N x 64]
              +-------+-------+
              |               |
          h_s [64]        h_t [64]      seed edge features e_seed {20|81|61}
              |               |             | (raw values, skip MP)
              +-------+-------+             |
                      v                     |
        concat [ h_s || h_t || e_seed ] <---+     width: 64+64+d_seed
                      |
        Linear(128+d_seed -> 64) . ReLU . Dropout . Linear 64->1
                      |
             sigmoid -> P(laundering)
```
Amber rule of the experiment: only `e_uv` width, `e_seed` width (and hence the two input
projections) change between configurations — every other weight shape is identical.

In [ ]:
def gin_mlp(hidden):
    '''The MLP "phi" applied after aggregation in every GIN layer.

    Identical in all configurations: Linear 64->64 . BatchNorm . ReLU . Linear 64->64.
    GIN theory needs at least one hidden layer here to keep WL expressiveness;
    BatchNorm tames sum-aggregated activations of hub accounts (degree up to 168k).
    '''
    return nn.Sequential(nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
                         nn.Linear(hidden, hidden))


def make_conv(hidden, mp_edge_dim):
    '''One graph convolution; this choice is the ONLY difference between configs.

    mp_edge_dim = 0  -> GINConv : message u->v is  h_u
    mp_edge_dim > 0  -> GINEConv: message u->v is  ReLU(h_u + W_e @ e_uv),
                        with W_e: [hidden, mp_edge_dim] the only extra weight.
    train_eps=True makes eps in (1+eps)*h_v learnable (self vs neighbourhood weight).
    '''
    if mp_edge_dim > 0:
        return GINEConv(gin_mlp(hidden), train_eps=True, edge_dim=mp_edge_dim)
    return GINConv(gin_mlp(hidden), train_eps=True)


class GINLayer(nn.Module):
    '''One message-passing step, optionally in both directions, with a residual.

    Update:   h_v <- h_v + Dropout(ReLU( conv(h, edges) ))
    Conv:     phi( (1+eps)*h_v + sum of messages from neighbours )

    edge_index rows are [senders; receivers] and messages flow sender -> receiver,
    so conv_in makes each account aggregate the accounts that PAID it. With
    bidirectional=True a second conv runs on flipped edges ("whom I paid") and
    the two 64-dim views are merged back to 64.

    Args:
        hidden        (int)  : embedding width (64)
        mp_edge_dim   (int)  : edge feature width in message passing (0 | 20 | 81)
        bidirectional (bool) : add the reverse-direction conv
        dropout       (float): dropout rate after the conv (0.3)
    '''

    def __init__(self, hidden, mp_edge_dim, bidirectional, dropout):
        super().__init__()
        self.use_edges = mp_edge_dim > 0
        self.bidirectional = bidirectional
        self.conv_in = make_conv(hidden, mp_edge_dim)          # "who paid me"
        if bidirectional:
            self.conv_out = make_conv(hidden, mp_edge_dim)     # "whom I paid"
            self.merge = nn.Linear(2 * hidden, hidden)         # concat 64||64 -> 64
        self.dropout = nn.Dropout(dropout)

    def apply_conv(self, conv, h, edge_index, edge_attr):
        '''Route the call: GINEConv takes edge features, GINConv takes none.'''
        if self.use_edges:
            return conv(h, edge_index, edge_attr)
        return conv(h, edge_index)

    def forward(self, h, edge_index, edge_attr):
        '''h: [N, 64] in -> [N, 64] out (refined through the residual).'''
        out = self.apply_conv(self.conv_in, h, edge_index, edge_attr)
        if self.bidirectional:
            # flip(0) swaps senders<->receivers = reversed edges; same edge_attr,
            # a transaction's features describe it in both directions.
            out_rev = self.apply_conv(self.conv_out, h, edge_index.flip(0), edge_attr)
            out = self.merge(torch.cat([out, out_rev], dim=1))
        return h + self.dropout(torch.relu(out))   # residual: refine, don't replace


class GIN(nn.Module):
    '''GIN edge classifier with a fixed architecture.

    Args:
        node_dim         (int)  : raw node feature width (6, entity one-hot)
        mp_edge_dim      (int)  : edge features in message passing (0 | 20 | 81)
        readout_edge_dim (int)  : edge features at the classifier (20 | 81 | 61)
        bidirectional    (bool) : incoming-only vs both-directions aggregation

    Parameter shapes (what "fixed architecture" means concretely):
        node_proj              : [64, node_dim] + [64]
        per layer  - conv MLP  : [64,64]+[64] . BN [64]+[64] . [64,64]+[64], eps scalar
        per layer  - GINE only : W_e [64, mp_edge_dim] + [64]
        per layer  - bidi only : second conv (same shapes) + merge [64, 128] + [64]
        classifier             : [64, 128 + readout_edge_dim] + [64], then [1, 64] + [1]
    Only W_e and the classifier's first layer may change with the feature set.
    '''

    def __init__(self, node_dim, mp_edge_dim, readout_edge_dim, bidirectional):
        super().__init__()
        # embed the 6 entity-type bits into the shared 64-dim working space
        self.node_proj = nn.Linear(node_dim, HIDDEN)
        # NUM_LAYERS (=2) layers of identical structure, each with its own weights
        self.layers = nn.ModuleList(
            [GINLayer(HIDDEN, mp_edge_dim, bidirectional, DROPOUT) for _ in range(NUM_LAYERS)])
        # readout MLP on [h_src || h_dst || raw seed edge features] -> one logit
        self.classifier = nn.Sequential(
            nn.Linear(2 * HIDDEN + readout_edge_dim, HIDDEN), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN, 1))

    def forward(self, x, edge_index, edge_attr, seed_index, seed_edge_feats):
        '''Score a batch of seed transactions.

        Args (all batch-local, produced by LinkNeighborLoader):
            x               [N, 6]       node features of the sampled subgraph
            edge_index      [2, E]       sampled edges (context + seeds)
            edge_attr       [E, mp_dim]  MP edge features, or None when MP = 'none'
            seed_index      [2, B]       endpoint ids of the B seed transactions
            seed_edge_feats [B, ro_dim]  raw readout features of the seed transactions

        Returns:
            [B] raw logits — BCEWithLogitsLoss / torch.sigmoid apply the sigmoid.
        '''
        # 1) node embeddings by message passing: [N, 6] -> [N, 64], 2 hops of context
        h = torch.relu(self.node_proj(x))
        for layer in self.layers:
            h = layer(h, edge_index, edge_attr)
        # 2) classify each seed s->t from its endpoints + its own raw features
        z = torch.cat([h[seed_index[0]], h[seed_index[1]], seed_edge_feats], dim=1)
        return self.classifier(z).squeeze(-1)      # [B, 1] -> [B]


## 4. Shared helpers — loader, prediction, metrics

Everything is parameterized by the config's column lists, so every configuration runs through
literally the same code. Seed edges' readout features are looked up via `batch.input_id`.

In [ ]:
def make_loader(g, shuffle, mp_idx, readout_idx):
    # graph used for message passing: node features + (optionally) the MP edge columns
    if len(mp_idx) > 0:
        mp_edge_attr = g.edge_attr[:, mp_idx]
    else:
        mp_edge_attr = None
    mp_graph = Data(x=g.x, edge_index=g.edge_index, edge_attr=mp_edge_attr,
                    edge_time=g.edge_time, num_nodes=g.num_nodes)

    # seed edges = the evaluated edges of this split
    seed_mask  = g.eval_mask
    seed_index = g.edge_index[:, seed_mask]
    seed_y     = g.y[seed_mask].float()
    seed_feats = g.edge_attr[seed_mask][:, readout_idx]      # readout features per seed edge

    if TEMPORAL_SAMPLING:
        loader = LinkNeighborLoader(mp_graph, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                                    edge_label_index=seed_index, edge_label=seed_y, shuffle=shuffle,
                                    time_attr='edge_time', edge_label_time=g.edge_time[seed_mask])
    else:
        loader = LinkNeighborLoader(mp_graph, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                                    edge_label_index=seed_index, edge_label=seed_y, shuffle=shuffle)
    return loader, seed_feats


def forward_batch(model, batch, seed_feats):
    # batch.input_id = positions of this batch's seed edges in the seed list
    batch = batch.to(device)
    feats = seed_feats[batch.input_id.cpu()].to(device)
    logits = model(batch.x, batch.edge_index, batch.edge_attr, batch.edge_label_index, feats)
    return logits, batch.edge_label


@torch.no_grad()
def predict(model, g, name, mp_idx, readout_idx):
    # true labels and predicted probabilities for all evaluated edges of a graph
    model.eval()
    loader, seed_feats = make_loader(g, shuffle=False, mp_idx=mp_idx, readout_idx=readout_idx)
    labels, probs = [], []
    for batch in tqdm(loader, desc=f'predict {name}', leave=False):
        logits, y = forward_batch(model, batch, seed_feats)
        probs.append(torch.sigmoid(logits).cpu().numpy())
        labels.append(y.cpu().numpy())
    return np.concatenate(labels).astype(int), np.concatenate(probs)


def best_f1_threshold(y, p):
    # threshold that maximises F1 on the given (validation) predictions
    precision, recall, thresholds = precision_recall_curve(y, p)
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    return float(thresholds[f1.argmax()]), float(f1.max())


def weighted_bce(y, p):
    # the training objective computed from eval-mode predictions, so the
    # train and validation loss curves are directly comparable
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return float(-np.mean(POS_WEIGHT * y * np.log(p) + (1 - y) * np.log(1 - p)))


def compute_metrics(y, p, threshold):
    pred = (p >= threshold).astype(int)
    return {
        'f1':        f1_score(y, pred),
        'precision': precision_score(y, pred, zero_division=0),
        'recall':    recall_score(y, pred),
        'pr_auc':    average_precision_score(y, p),
        'roc_auc':   roc_auc_score(y, p),
        'threshold': threshold,
    }

## 5. One experiment = one function call

`run_experiment(mp, readout)` builds a fresh model, trains it with the shared recipe, keeps the
best-validation-F1 checkpoint, scores all three splits at the validation-chosen threshold, saves
everything under `outputs/<run_name>/`, and returns a summary row.

In [ ]:
def run_experiment(mp, readout):
    '''Train and evaluate one feature configuration; returns a summary dict.'''
    run_name = f'gin_mp-{mp}_readout-{readout}_dir-{MP_DIRECTION}'
    if TEMPORAL_SAMPLING:
        run_name = run_name + '_temporal'
    out_dir = f'{OUT_ROOT}/{run_name}'
    os.makedirs(out_dir, exist_ok=True)
    mp_idx, readout_idx = COL_SETS[mp], COL_SETS[readout]

    # same seed for every run -> identical init of the shared (invariant) weights
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    model = GIN(NODE_DIM, len(mp_idx), len(readout_idx), MP_DIRECTION == 'bidirectional').to(device)
    n_params = sum(p.numel() for p in model.parameters())
    width_dependent = 0
    for pname, p in model.named_parameters():
        if '.lin.' in pname or pname.startswith('classifier.0.'):
            width_dependent += p.numel()
    invariant_params = n_params - width_dependent

    print(f'=== {run_name} ===')
    print(f'MP edge features: {len(mp_idx)} | readout edge features: {len(readout_idx)}')
    print(f'total params: {n_params:,} | architecture-invariant: {invariant_params:,}')

    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=device))
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    history = []
    best_val_f1, best_epoch, best_threshold = -1.0, 0, 0.5

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()

        # ---- one training pass ----
        model.train()
        loader, seed_feats = make_loader(train_graph, shuffle=True,
                                         mp_idx=mp_idx, readout_idx=readout_idx)
        for batch in tqdm(loader, desc='train', leave=False):
            optimizer.zero_grad()
            logits, y = forward_batch(model, batch, seed_feats)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

        # ---- score train and validation in eval mode (overfitting diagnostics) ----
        y_train, p_train = predict(model, train_graph, 'train', mp_idx, readout_idx)
        y_val,   p_val   = predict(model, val_graph,   'val',   mp_idx, readout_idx)

        # threshold is always chosen on validation; train F1 reuses it
        threshold, val_f1 = best_f1_threshold(y_val, p_val)
        train_f1 = f1_score(y_train, (p_train >= threshold).astype(int))

        history.append({
            'epoch':        epoch,
            'train_loss':   weighted_bce(y_train, p_train),
            'val_loss':     weighted_bce(y_val, p_val),
            'train_f1':     train_f1,
            'val_f1':       val_f1,
            'train_pr_auc': average_precision_score(y_train, p_train),
            'val_pr_auc':   average_precision_score(y_val, p_val),
            'val_roc_auc':  roc_auc_score(y_val, p_val),
            'threshold':    threshold,
            'seconds':      time.time() - t0,
        })
        r = history[-1]
        print(f"epoch {epoch:02d} | loss {r['train_loss']:.4f} / {r['val_loss']:.4f} | "
              f"F1 {r['train_f1']:.4f} / {r['val_f1']:.4f} @ {threshold:.3f} | "
              f"PR-AUC {r['train_pr_auc']:.4f} / {r['val_pr_auc']:.4f} | "
              f"{r['seconds']:.0f}s   (train / val)")

        if val_f1 > best_val_f1:
            best_val_f1, best_epoch, best_threshold = val_f1, epoch, threshold
            torch.save(model.state_dict(), f'{out_dir}/best.pt')

        scheduler.step()

    pd.DataFrame(history).to_csv(f'{out_dir}/history.csv', index=False)
    print(f'best validation F1 {best_val_f1:.4f} at epoch {best_epoch} (threshold {best_threshold:.3f})')

    # ---- final evaluation: best checkpoint, one validation-chosen threshold ----
    model.load_state_dict(torch.load(f'{out_dir}/best.pt', map_location=device))
    y_train, p_train = predict(model, train_graph, 'train', mp_idx, readout_idx)
    y_val,   p_val   = predict(model, val_graph,   'val',   mp_idx, readout_idx)
    y_test,  p_test  = predict(model, test_graph,  'test',  mp_idx, readout_idx)
    train_metrics = compute_metrics(y_train, p_train, best_threshold)
    val_metrics   = compute_metrics(y_val,   p_val,   best_threshold)
    test_metrics  = compute_metrics(y_test,  p_test,  best_threshold)

    results = {
        'run': run_name, 'mp_edge_feats': mp, 'readout_edge_feats': readout,
        'mp_direction': MP_DIRECTION, 'temporal_sampling': TEMPORAL_SAMPLING,
        'architecture': {'hidden': HIDDEN, 'layers': NUM_LAYERS, 'dropout': DROPOUT,
                         'neighbors': NUM_NEIGHBORS, 'params': n_params,
                         'invariant_params': invariant_params},
        'best_epoch': best_epoch, 'seed': SEED,
        'train': train_metrics, 'val': val_metrics, 'test': test_metrics,
    }
    json.dump(results, open(f'{out_dir}/results.json', 'w'), indent=2)
    print(pd.DataFrame({'train': train_metrics, 'validation': val_metrics, 'test': test_metrics}).round(4))

    # ---- diagnostic curves ----
    df = pd.DataFrame(history)
    fig, ax = plt.subplots(2, 2, figsize=(13, 8))
    fig.suptitle(run_name, fontweight='bold')
    ax[0, 0].plot(df['epoch'], df['train_loss'], marker='o', label='train')
    ax[0, 0].plot(df['epoch'], df['val_loss'], marker='s', label='validation')
    ax[0, 0].set_title('weighted BCE loss'); ax[0, 0].set_xlabel('epoch'); ax[0, 0].legend()
    ax[0, 1].plot(df['epoch'], df['train_f1'], marker='o', label='train')
    ax[0, 1].plot(df['epoch'], df['val_f1'], marker='s', label='validation')
    ax[0, 1].set_title('F1 (at the validation-chosen threshold)'); ax[0, 1].set_xlabel('epoch'); ax[0, 1].legend()
    ax[1, 0].plot(df['epoch'], df['train_pr_auc'], marker='o', label='train')
    ax[1, 0].plot(df['epoch'], df['val_pr_auc'], marker='s', label='validation')
    ax[1, 0].set_title('PR-AUC'); ax[1, 0].set_xlabel('epoch'); ax[1, 0].legend()
    precision, recall, _ = precision_recall_curve(y_test, p_test)
    ax[1, 1].plot(recall, precision, label=f'test AP = {test_metrics["pr_auc"]:.3f}')
    ax[1, 1].axhline(y_test.mean(), color='gray', linestyle='--', label='random')
    ax[1, 1].set_title('test precision-recall'); ax[1, 1].set_xlabel('recall'); ax[1, 1].set_ylabel('precision'); ax[1, 1].legend()
    plt.tight_layout()
    plt.savefig(f'{out_dir}/curves.png', dpi=120)
    plt.show()

    # flat summary row for the comparison table
    return {'run': run_name, 'mp': mp, 'readout': readout, 'best_epoch': best_epoch,
            'val_f1': val_metrics['f1'], 'test_f1': test_metrics['f1'],
            'test_precision': test_metrics['precision'], 'test_recall': test_metrics['recall'],
            'test_pr_auc': test_metrics['pr_auc'], 'test_roc_auc': test_metrics['roc_auc'],
            'params': n_params, 'invariant_params': invariant_params}

## 6. Run the batch

In [ ]:
all_results = []
for cfg in CONFIGS:
    print('=' * 90)
    all_results.append(run_experiment(**cfg))
print('=' * 90)
print('batch complete')

## 7. Comparison across the batch

All runs share the same architecture — `invariant_params` must be identical in every row.

In [ ]:
summary = pd.DataFrame(all_results)
summary.to_csv(f'{OUT_ROOT}/batch_summary.csv', index=False)
summary.round(4)

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/outputs_gin_baseline_191026', 'zip', '/kaggle/working/outputs')